# 🧬 Chai-1 Structure Prediction Example

This notebook demonstrates how to predict protein and protein-ligand complex structures using **Chai-1**.

## 📖 What is Chai-1?

Chai-1 is a multi-modal foundation model for molecular structure prediction from Chai Discovery. It can predict structures of proteins, ligands, and glycans, and their complexes.

### Key Features:
- **Multi-modal** -- Supports proteins, ligands, and glycans
- **Complex prediction** -- Handles protein-ligand and multi-chain complexes
- **Confidence scores** -- Provides pLDDT, pTM, and ipTM metrics
- **MSA-optional** -- Can run with or without multiple sequence alignments

## 📥 Imports

## 📦 Shared Data Types

### `StructurePredictionComplex`
A biological complex containing one or more molecular chains to predict together.

| Field | Type | Description |
|-------|------|-------------|
| `chains` | `List[Chain]` | Chains in the complex. Accepts strings, `Chain` objects, or dicts |

### `Chain`
A single molecular chain.

| Field | Type | Description |
|-------|------|-------------|
| `sequence` | `str` | Sequence (protein amino acids, DNA/RNA bases, or ligand SMILES) |
| `entity_type` | `Optional[str]` | `"protein"`, `"dna"`, `"rna"`, or `"ligand"`. Auto-inferred if `None` |

### `Structure`
A predicted 3D structure with coordinates, confidence metrics, and export methods.

## 📥 Imports

In [1]:
from bio_programming_tools import (
    Chai1Config,
    Chai1Input,
    Chain,
    StructurePredictionComplex,
    run_chai1,
)
from pathlib import Path

## 🔍 Protein-Ligand Complex Prediction (MfnG Example)

Let's predict the structure of MfnG protein bound to L-tyrosine ligand.

### API Reference

**`Chai1Input`**

| Field | Type | Description |
|-------|------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | List of complexes to predict structures for. Total length across all chains must not exceed 2,048 residues |

**`Chai1Config`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `use_esm_embeddings` | `bool` | `True` | Whether to use ESM embeddings for improved predictions |
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains using ColabFold search |
| `num_trunk_recycles` | `int` | `3` | Number of iterative refinement passes through trunk network |
| `num_diffn_timesteps` | `int` | `200` | Number of denoising steps in the diffusion process |
| `num_diffn_samples` | `int` | `1` | Number of independent structure samples per complex (only best is returned) |
| `num_trunk_samples` | `int` | `1` | Number of independent trunk forward passes per diffusion sample |

**`Chai1Output`**

| Field | Type | Description |
|-------|------|-------------|
| `structures` | `List[Structure]` | List of predicted structures, one per input complex |

In [2]:
# MfnG protein sequence
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = Chai1Input(complexes=[complex])

# Configure Chai-1
config = Chai1Config(
    verbose=True,
    device="cuda",  # Change to "cpu" if no GPU available
    use_esm_embeddings=True,
    use_msa=False,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    num_diffn_samples=1,
)

# Run prediction
result = run_chai1(inputs, config)
mfng_structure = result.structures[0]

# Print metrics
print(f"\n✅ Structure predicted!")
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Average pLDDT: {mfng_structure.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.iptm:.3f}")

Folding structures (Chai-1):   0%|          | 0/1 [00:00<?, ?complex/s]

Diffusion steps: 100%|██████████| 199/199 [00:18<00:00, 10.52it/s]


Score=0.6840, writing output to /tmp/tmpia69sn83/output/pred.model_idx_0.cif


Folding structures (Chai-1): 100%|██████████| 1/1 [01:13<00:00, 73.16s/complex]


✅ Structure predicted!
  Number of chains: 2
  Protein length: 384 residues
  Average pLDDT: 0.838
  pTM score: 0.847
  ipTM score: 0.643


### 🎨 Visualize MfnG-Ligand Complex

In [3]:
mfng_structure.visualize(style='stick', color_by='chain')

## 💾 Export Results

Save predicted structures for further analysis in other tools like PyMOL, ChimeraX, or VMD.

### Supported formats:
- **PDB** -- Standard protein data bank format, widely compatible
- **mmCIF** -- Modern crystallographic information file, supports larger structures

The B-factor column contains the pLDDT scores for confidence visualization.

In [ ]:
# Create output directory
output_dir = Path("./chai_outputs")
output_dir.mkdir(exist_ok=True)

# Export results to pdb files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")